# Part 3: Saved Epoch Validation
**Role:** Person 1 (EEG Data Loading and Preprocessing)  
**Project:** Deep Learning-Based Motor Imagery EEG Classification for a Simulated Real-Time BCI

The saved epochs use these event-code and class-label mappings:

| Event code | Class label | Motor imagery class |
| --- | ---: | --- |
| 769 | 0 | Left Hand |
| 770 | 1 | Right Hand |
| 771 | 2 | Both Feet |
| 772 | 3 | Tongue |

The cell below loads the existing `A01T_epochs.npz` and validates its arrays without reopening the raw GDF or rerunning epoch extraction.

In [ ]:
from pathlib import Path
import sys
import numpy as np


def find_project_root(start_path: Path) -> Path:
    for candidate in (start_path, *start_path.parents):
        if (candidate / "src").is_dir() and (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find project root above {start_path}; expected src/ and data/processed/."
    )


workspace_root = find_project_root(Path.cwd().resolve())
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from src.extract_epochs import CLASS_NAMES, EVENT_MAPPING, validate_xy

epochs_path = workspace_root / "data" / "processed" / "A01T_epochs.npz"
print(f"Loading saved epochs from {epochs_path.name}...")
with np.load(epochs_path, allow_pickle=False) as archive:
    X = archive["X"]
    y = archive["y"]
    channel_names = archive["channel_names"].tolist()

expected_classes = tuple(EVENT_MAPPING.values())
print("Validating saved X and y arrays...")
class_counts = validate_xy(X, y, expected_classes=expected_classes)
sample_label_counts_match = X.shape[0] == y.shape[0]

print(f"Epoch archive: {epochs_path.name}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"EEG channels: {X.shape[1]} (channel names: {len(channel_names)})")
print("Finite X values: True (validated by validate_xy)")
print(f"X and y sample counts match: {sample_label_counts_match}")
print("Class counts:")
for event_code, class_label in EVENT_MAPPING.items():
    print(
        f"  Event {event_code} -> class {class_label} "
        f"({CLASS_NAMES[class_label]}): {class_counts[class_label]}"
    )

Epoch archive: A01T_epochs.npz
X shape: (288, 22, 1000)
y shape: (288,)
EEG channels: 22 (channel names: 22)
Finite X values: True
X and y sample counts match: True
Class counts:
  Event 769 -> class 0 (Left Hand): 72
  Event 770 -> class 1 (Right Hand): 72
  Event 771 -> class 2 (Both Feet): 72
  Event 772 -> class 3 (Tongue): 72
